# AOI Tool — Debug Notebook
Inspect all backend data served to the frontend.

**Requirements:**
- Backend running at `http://127.0.0.1:PORT` (set `BASE_URL` below)
- At least one AP sync run (`POST /api/sync/run?sync_type=ap`)

Run cells top to bottom.

## 0 — Configuration

In [ ]:
import requests
import json
import pandas as pd
from IPython.display import display, HTML

BASE_URL = "http://127.0.0.1:8000"   # <- change port if needed
DB_PATH  = r"C:/Users/username/Documents/aoi-web/aoi_tool.db"  # <- adjust

def get(path, **params):
    r = requests.get(f'{BASE_URL}{path}', params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def color_row(val):
    c = {'red':'#ffcccc','orange':'#ffe0b2','yellow':'#fff9c4','green':'#e8f5e9'}
    return f'background-color: {c.get(val, "white")}'

def color_type(val):
    c = {'Critical':'#ffcccc','Suggestion':'#ffe0b2','Info':'#e3f2fd'}
    return f'background-color: {c.get(val, "white")}'

print(f'Connected to: {BASE_URL}')

## 1 — Server Status (`/api/status`)

In [ ]:
status = get('/api/status')
print(json.dumps(status, indent=2))

## 2 — Sync Status (`/api/sync/status`)

In [ ]:
sync = get('/api/sync/status')
rows = []
for t in ('pp','cli','ap','pm_type'):
    s = sync.get(t)
    if s:
        rows.append({'type':t,'status':s.get('status'),'total':s.get('total'),
            'changed':s.get('changed'),'new':s.get('new'),'deleted':s.get('deleted'),
            'duration_s':s.get('duration_s'),'started_at':s.get('started_at'),
            'finished_at':s.get('finished_at'),'error_msg':s.get('error_msg')})
    else:
        rows.append({'type':t,'status':'never run'})
display(pd.DataFrame(rows))
running = sync.get('running',[])
print(f'Running: {[r["sync_type"] for r in running]}' if running else 'No sync running')

## 3 — AP Response (`/api/ap`)
Full inspection result served to the frontend Inspektion tab.

In [ ]:
try:
    ap = get('/api/ap')
    print(f"inspection_type : {ap['inspection_type']}")
    print(f"timestamp       : {ap['timestamp']}")
    print(f"duration_seconds: {ap['duration_seconds']}")
    print(f"BG count        : {ap['count']}")
except Exception as e:
    print(f'Error: {e} — run AP sync first')
    ap = None

### 3a — BG summary table

In [ ]:
if ap:
    rows = []
    for bg in ap['results']:
        rows.append({'name':bg['name'],'kunde':bg['kunde'],'lp_nr':bg['lp_nr'],
            'dmc':bg['dmc'],'medi':bg['medi'],'aoi_color':bg['aoi_color'],
            'bg_color':bg['bg_color'],'smd_line':bg['smd_line'],
            'auftragsmenge':bg['auftragsmenge'],
            'pp_count':len(bg['pp_list']),'bg_errors':len(bg['bg_errors']),
            'row_color':bg['row_color']})
    df = pd.DataFrame(rows)
    display(df.style.applymap(color_row, subset=['row_color']))

### 3b — PP detail table

In [ ]:
if ap:
    rows = []
    for bg in ap['results']:
        for pp in bg['pp_list_detail']:
            rows.append({'bg_name':bg['name'],'pp_name':pp['name'],
                'side':pp['side'],'locked':pp['locked'],'cli':pp['cli'],
                'nutzen_in_lp':pp['nutzen_in_lp'],'pm_count':pp['pm_count'],
                'hinweis':pp['hinweis'],'pp_errors':len(pp['errors']),
                'row_color':pp['row_color']})
    df = pd.DataFrame(rows)
    display(df.style.applymap(color_row, subset=['row_color']))

### 3c — All errors

In [ ]:
if ap:
    rows = []
    for bg in ap['results']:
        for e in bg['bg_errors']:
            rows.append({'level':'BG',**e})
        for pp in bg['pp_list_detail']:
            for e in pp['errors']:
                rows.append({'level':'PP',**e})
    if rows:
        df = pd.DataFrame(rows)[['level','bg_name','pp_name','error_code',
                                  'error_type','short_desc','long_desc']]
        display(df.style.applymap(color_type, subset=['error_type']))
    else:
        print('No errors found')

## 4 — DB Tables (direct SQLite read)

In [ ]:
import sqlite3
conn_db = sqlite3.connect(DB_PATH)
conn_db.row_factory = sqlite3.Row

def table(sql, params=()):
    rows = conn_db.execute(sql, params).fetchall()
    return pd.DataFrame([dict(r) for r in rows])

print('DB connected:', DB_PATH)

### 4a — BG table

In [ ]:
df = table('SELECT * FROM bg ORDER BY bg_name LIMIT 100')
display(df)
n = table('SELECT COUNT(*) as n FROM bg').iloc[0]['n']
print(f'Total BG in DB: {n}')

### 4b — PP table

In [ ]:
df = table('SELECT pp_name, locked, cli, nutzen_in_lp, oldest_mod, mtime_cad, mtime_desc, synced_at FROM pp ORDER BY pp_name LIMIT 100')
display(df)
n = table('SELECT COUNT(*) as n FROM pp').iloc[0]['n']
print(f'Total PP in DB: {n}')

### 4c — PP PM table

In [ ]:
df = table('SELECT pp_name, pm_name, ihl_nr, pm_type FROM pp_pm ORDER BY pp_name, pm_name LIMIT 200')
display(df)
n = table('SELECT COUNT(*) as n FROM pp_pm').iloc[0]['n']
print(f'Total pp_pm rows: {n}')
print('\npm_type distribution:')
display(table('SELECT pm_type, COUNT(*) as count FROM pp_pm GROUP BY pm_type'))

### 4d — CLI Global table

In [ ]:
df = table('SELECT pm_name, cli, active_macros, last_modified_mac_name, mtime_cle, synced_at FROM cli_global ORDER BY cli, pm_name LIMIT 100')
display(df)
n = table('SELECT COUNT(*) as n FROM cli_global').iloc[0]['n']
print(f'Total cli_global rows: {n}')

### 4e — CLI Local table

In [ ]:
df = table('SELECT pm_name, pp_name, cli, active_macros, last_modified_mac_name, synced_at FROM cli_local ORDER BY pp_name, pm_name LIMIT 100')
display(df)
n = table('SELECT COUNT(*) as n FROM cli_local').iloc[0]['n']
print(f'Total cli_local rows: {n}')

### 4f — Errors table

In [ ]:
df = table('SELECT * FROM error ORDER BY bg_name, pp_name LIMIT 200')
if not df.empty:
    display(df.style.applymap(color_type, subset=['error_type']))
else:
    print('No errors in DB')
print('\nError distribution:')
display(table('SELECT error_code, error_type, COUNT(*) as count FROM error GROUP BY error_code, error_type ORDER BY count DESC'))

### 4g — Sync log

In [ ]:
display(table('SELECT * FROM sync_log ORDER BY id DESC LIMIT 30'))

## 5 — PM Search (`/api/search/pm`)

In [ ]:
PM_QUERY    = "widerstand"   # <- change
SEARCH_TYPE = "contains"     # exact | contains | starts_with | ends_with
CLI_FILTER  = "all"

result = get('/api/search/pm', q=PM_QUERY, search_type=SEARCH_TYPE, cli=CLI_FILTER)
print(f"Query: '{result['query']}' ({result['search_type']}, cli={result['cli_filter']})")
print(f"Results: {result['count']}")
if result['count'] > 0:
    display(pd.DataFrame(result['results']).head(50))

### 5a — CLI list

In [ ]:
cli_data = get('/api/search/cli-list')
print(f"Available CLIs ({len(cli_data['cli_list'])}):")
for c in cli_data['cli_list']:
    print(f'  {c}')

## 6 — Custom DB Queries

In [ ]:
# PMs missing from CLI (Critical errors)
df = table(
    'SELECT p.pp_name, p.cli, pm.pm_name, pm.ihl_nr '
    'FROM pp_pm pm JOIN pp p ON p.pp_name = pm.pp_name '
    'WHERE pm.pm_type IS NULL ORDER BY p.pp_name, pm.pm_name'
)
print(f'PMs missing from CLI: {len(df)}')
display(df)

In [ ]:
# PP with oldest .mod exceptions
df = table(
    'SELECT pp_name, cli, oldest_mod, mtime_mod FROM pp '
    'WHERE oldest_mod IS NOT NULL ORDER BY oldest_mod ASC LIMIT 20'
)
display(df)

In [ ]:
# BG with most errors
df = table(
    'SELECT bg_name, COUNT(*) as total, '
    'SUM(CASE WHEN error_type="Critical" THEN 1 ELSE 0 END) as critical, '
    'SUM(CASE WHEN error_type="Suggestion" THEN 1 ELSE 0 END) as suggestion, '
    'SUM(CASE WHEN error_type="Info" THEN 1 ELSE 0 END) as info '
    'FROM error GROUP BY bg_name ORDER BY critical DESC, total DESC LIMIT 20'
)
display(df)

## 7 — Cleanup

In [ ]:
conn_db.close()
print('DB connection closed')